# 01 Localizer V2: parameter sweep -> best preset -> ensemble
CLEAN_V2 loss = L_BCE + L_Tversky + λ_peak(t)·L_Peak with tunable parameters: BCE positive weight, Tversky (α_FP, β_FN),
peak-loss start / maximum weight / radius / temperature, and the decoder activation (ReLU / LeakyReLU / SiLU / GELU).
Step 1 tries each preset for a few epochs (validation hit-rate + Dice decide), step 2 trains the 3-seed ensemble with the best one,
step 3 picks the peak-smoothing sigma on validation and saves the ensemble heat-maps.

In [1]:
from google.colab import drive; drive.mount("/content/drive")
import glob, os, sys, json
hits = glob.glob("/content/drive/MyDrive/mammo_semcom/**/src/config.py", recursive=True)
CODE_DIR = os.path.dirname(os.path.dirname(hits[0])); WORK_DIR = "/content/drive/MyDrive/mammo_semcom_work"
sys.path.insert(0, CODE_DIR); print("code:", CODE_DIR)
import numpy as np, cv2, pandas as pd
from src.config import Cfg
from src import data
from src.train import localize, hit_rate, dice_np
from src.train_v2 import image_centres, train_localizer_v2, ensemble_heat, load_localizer_v2

SWEEP, SWEEP_EPOCHS, FINAL_EPOCHS, M_SEEDS = True, 8, 20, 3        # SWEEP=False -> use PRESET below directly
PRESET = "spec"
PRESETS = {
    "spec":        dict(),                                                                # CLEAN_V2 exactly as specified
    "peak_early":  dict(P=dict(peak=(2, .5, 3))),                                         # peak loss from epoch 2, max weight 0.5
    "peak_strong": dict(P=dict(peak=(2, 1.0, 4))),
    "tversky_fp":  dict(P=dict(tv_a=.6, tv_b=.4, peak=(2, .5, 3))),                       # penalise false positives more (fewer wrong peaks)
    "dice_like":   dict(P=dict(tv_a=.5, tv_b=.5, peak=(2, .5, 3))),
    "posw5":       dict(P=dict(pos_w=5., peak=(2, .5, 3))),
    "silu":        dict(P=dict(peak=(2, .5, 3)), act="silu"),
    "leaky":       dict(P=dict(peak=(2, .5, 3)), act="leaky"),
}
cfg = Cfg(work_dir=WORK_DIR, loc_epochs=FINAL_EPOCHS).ensure()
images, masks, images_df, abn_df = data.load_cache(cfg)
centres = image_centres(abn_df, len(images))
idx = {s: np.where(images_df.split.values == s)[0] for s in ("train", "val", "test")}
print({k: len(v) for k, v in idx.items()})

Mounted at /content/drive
code: /content/drive/MyDrive/mammo_semcom
{'train': 2190, 'val': 455, 'test': 458}


## Step 1: parameter sweep (one seed, short)

In [2]:
if SWEEP:
    res = {}
    for k, v in PRESETS.items():
        _, hr, dc = train_localizer_v2(cfg, images, masks, centres, idx["train"], idx["val"], cfg.seed, None, v.get("P"), v.get("act", "relu"), SWEEP_EPOCHS, verbose=False)
        res[k] = dict(val_hit=hr, val_dice=dc, score=hr + dc); print(k, {a: round(b, 3) for a, b in res[k].items()})
    table = pd.DataFrame(res).T.sort_values("score", ascending=False); display(table.round(3))
    PRESET = table.index[0]
print("using preset:", PRESET, PRESETS[PRESET])

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 193MB/s]


spec {'val_hit': 0.598, 'val_dice': 0.324, 'score': 0.922}
peak_early {'val_hit': 0.593, 'val_dice': 0.289, 'score': 0.882}
peak_strong {'val_hit': 0.587, 'val_dice': 0.293, 'score': 0.88}
tversky_fp {'val_hit': 0.604, 'val_dice': 0.317, 'score': 0.921}
dice_like {'val_hit': 0.587, 'val_dice': 0.299, 'score': 0.886}
posw5 {'val_hit': 0.607, 'val_dice': 0.289, 'score': 0.896}
silu {'val_hit': 0.609, 'val_dice': 0.306, 'score': 0.915}
leaky {'val_hit': 0.602, 'val_dice': 0.311, 'score': 0.913}


,val_hit,val_dice,score
spec,0.598,0.324,0.922
tversky_fp,0.604,0.317,0.921
silu,0.609,0.306,0.915
leaky,0.602,0.311,0.913
posw5,0.607,0.289,0.896
dice_like,0.587,0.299,0.886
peak_early,0.593,0.289,0.882
peak_strong,0.587,0.293,0.880


using preset: spec {}


## Step 2: final ensemble with the chosen preset

In [3]:
pr = PRESETS[PRESET]; act = pr.get("act", "relu")
localizers = [train_localizer_v2(cfg, images, masks, centres, idx["train"], idx["val"], cfg.seed + s, f"v2_s{s}", pr.get("P"), act)[0] for s in range(M_SEEDS)]

seed 42 ep  1/20 | loss 1.367 | val hit 0.387 | val Dice 0.200
seed 42 ep  2/20 | loss 1.035 | val hit 0.398 | val Dice 0.175
seed 42 ep  3/20 | loss 1.002 | val hit 0.407 | val Dice 0.201
seed 42 ep  4/20 | loss 1.007 | val hit 0.481 | val Dice 0.231
seed 42 ep  5/20 | loss 0.981 | val hit 0.545 | val Dice 0.279
seed 42 ep  6/20 | loss 1.478 | val hit 0.589 | val Dice 0.304
seed 42 ep  7/20 | loss 1.070 | val hit 0.549 | val Dice 0.240
seed 42 ep  8/20 | loss 1.887 | val hit 0.567 | val Dice 0.308
seed 42 ep  9/20 | loss 1.956 | val hit 0.567 | val Dice 0.325
seed 42 ep 10/20 | loss 1.400 | val hit 0.563 | val Dice 0.307
seed 42 ep 11/20 | loss 0.558 | val hit 0.549 | val Dice 0.284
seed 42 ep 12/20 | loss 0.723 | val hit 0.567 | val Dice 0.297
seed 42 ep 13/20 | loss 0.795 | val hit 0.538 | val Dice 0.288
seed 42 ep 14/20 | loss 0.744 | val hit 0.560 | val Dice 0.280
seed 42 ep 15/20 | loss 0.873 | val hit 0.576 | val Dice 0.296
seed 42 ep 16/20 | loss 0.454 | val hit 0.552 | val Dic

## Step 3: heat-maps, peak sigma (validation), evaluation

In [4]:
heat = ensemble_heat(localizers, images)
np.save(cfg.path("cache", "heatmaps_v2.npy"), heat)
ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (49, 49))
def hit_sigma(ids, sigma):
    out = []
    for i in ids:
        cy, cx, _ = localize(heat[i], sigma); out.append(bool(cv2.dilate((masks[i] > 0).astype(np.uint8), ker)[int(round(cy)), int(round(cx))]))
    return float(np.mean(out))
sigma = max((3., 5., 8., 12., 16.), key=lambda s: hit_sigma(idx["val"], s))
thr = max(np.arange(.1, .95, .1), key=lambda t: dice_np(heat[idx["val"]], masks[idx["val"]], t))
for s in ("val", "test"):
    print(f"{s}: hit-rate {hit_sigma(idx[s], sigma):.3f} (sigma={sigma}) | Dice@{thr:.1f} {dice_np(heat[idx[s]], masks[idx[s]], thr):.3f}")
json.dump(dict(preset=PRESET, act=act, sigma=sigma, dice_thr=float(thr)), open(cfg.path("models", "loc_v2.json"), "w"))

val: hit-rate 0.631 (sigma=8.0) | Dice@0.4 0.335
test: hit-rate 0.581 (sigma=8.0) | Dice@0.4 0.303
